In [0]:
df = spark.sql(
  """
  select * from `workspace`.`bixi-fs`.`bixi-rides` 
  where end_station_name is not null
  limit 100
  """
)
display(df)

In [0]:
spark.sql("SHOW CATALOGS").show()

spark.sql("SHOW SCHEMAS IN `workspace`").show()

spark.sql("SHOW VOLUMES IN `workspace`.`bixi-fs`").show()

In [0]:
dbutils.fs.ls("/Volumes/workspace/bixi-fs/streaming/")

In [0]:
import requests
import json
import time
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

from pathlib import Path
Path().resolve()

url = "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"

def fetch_data():
    response = requests.get(url)
    data = response.json()
    
    stations = data["data"]["stations"]
    
    rows = [Row(**station) for station in stations]
    df = spark.createDataFrame(rows)
    df = df.withColumn("ingestion_time", current_timestamp())
    
    return df

# Create Delta table path
delta_path = "/Volumes/workspace/bixi-fs/streaming/"

# Simulate streaming input (poll 5 times)
for i in range(5):
    df = fetch_data()
    df.write.format("delta").mode("append").save(delta_path)
    print(f"Batch {i+1} written")
    time.sleep(10)  # wait 10 seconds

In [0]:
dbutils.fs.ls("/Volumes/workspace/bixi-fs/streaming/")

In [0]:
delta_path = "/Volumes/workspace/bixi-fs/streaming/bike_station_info"
checkpoint_path = "/Volumes/workspace/bixi-fs/checkpoints/read_stream_test"

import requests
from pyspark.sql.functions import current_timestamp

url = "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"

response = requests.get(url)
data = response.json()
stations = data["data"]["stations"]

df = spark.createDataFrame(stations)
df = df.withColumn("ingestion_time", current_timestamp())

df.write.format("delta").mode("overwrite").save(delta_path)

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

url = "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"

def fetch_and_write(batch_df, batch_id):
    response = requests.get(url)
    data = response.json()
    
    stations = data["data"]["stations"]
    
    df = spark.createDataFrame(stations)
    df = df.withColumn("ingestion_time", current_timestamp())
    
    (df.write
       .format("delta")
       .mode("append")
       .save(delta_path))
    
    print(f"Batch {batch_id} written with {len(stations)} stations")

import time

start_time = time.time()
duration = 30  # seconds

while time.time() - start_time < duration:

    query = (
        spark.readStream
             .format("rate")
             .option("rowsPerSecond", 1)
             .load()
             .writeStream
             .foreachBatch(fetch_and_write)
             .trigger(availableNow=True)
             .option("checkpointLocation", checkpoint_path)
             .start()
    )

    query.awaitTermination()
    time.sleep(5)  # wait before next poll

In [0]:
df = spark.read.format("delta").load(delta_path)
df.count()

display(df)

In [0]:
df = spark.sql("""
select * from `workspace`.`bixi-fs`.`bixi-rides` 
where end_station_name is not null
limit 100;
""")

df.show()